# Hybrid Search RAG — BM25 + Vector + Reciprocal Rank Fusion
## BM25 + Vectors + RRF — Why One Retriever Is Never Enough
⏱ ~15 min

Pure vector search is powerful — but it has a blind spot. Ask it for `"Apex-X200 read speed"` and it may return specs for the X100 because the embedding model treats similar-looking model numbers as semantically close. **Hybrid Search** fixes this by combining two complementary retrieval signals:

- **Dense (vector) search** — captures meaning, handles paraphrases and conceptual queries
- **Sparse (BM25/keyword) search** — captures exact tokens: model numbers, names, codes, legal references
- **Reciprocal Rank Fusion (RRF)** — merges both ranked lists into a single authoritative ranking

This is the production pattern used by Pinecone Hybrid, Weaviate, OpenSearch, and Elasticsearch's BM25+kNN mode.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why hybrid? Dense vs sparse blind spots |
| 2 | **BM25 Deep Dive** — How term frequency + inverse document frequency works |
| 3 | **Vector Retrieval** — Embeddings and semantic similarity recap |
| 4 | **Head-to-Head** — BM25 vs vector on the same queries |
| 5 | **Reciprocal Rank Fusion** — The math behind merging ranked lists |
| 6 | **EnsembleRetriever** — LangChain's hybrid wrapper in practice |
| 7 | **Full RAG Pipeline** — LangGraph graph: retrieve → generate |
| 8 | **Tuning Weights** — Calibrating BM25 vs vector balance for your domain |
| ★ | **Advanced: Reranking** — Cross-encoders on top of hybrid retrieval |

---

### Prerequisites
- Python 3.10+ with a `.venv` (or Google Colab)
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- **Recommended prerequisite:** Complete [12-basic-rag-notebook](../12-basic-rag-notebook/rag_workbook.ipynb) first to understand pure vector RAG before comparing here

### Key References
> Robertson, S. & Walker, S. (1994). *Some Simple Effective Approximations to the 2-Poisson Model for Probabilistic Weighted Retrieval.* SIGIR 1994. *(The original BM25 paper)*
> Robertson, S. & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in Information Retrieval.
> Cormack, G.V., Clarke, C.L.A., & Buettcher, S. (2009). *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods.* SIGIR 2009. https://dl.acm.org/doi/10.1145/1571941.1572114
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "chromadb",
            "rank_bm25",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import math
import os
from collections import Counter
from typing import List, TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), label = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, START, StateGraph

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running embedding or LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Why Hybrid? The Two Blind Spots

---

### Dense Retrieval (Vector Search) — Strengths and Weaknesses

Vector search converts text to embeddings and retrieves by cosine similarity. It is excellent at **semantic intent** — a query for `"affordable laptop for everyday use"` will match docs about `"budget notebooks for students"` even without shared keywords.

**But it has a critical blind spot:** model numbers, product codes, names, and rare identifiers are treated as nearly synonymous in embedding space because they co-occur in similar contexts. The model `Apex-X200` and `Apex-X100` are adjacent in vector space — but they are completely different products.

### Sparse Retrieval (BM25/Keyword) — Strengths and Weaknesses

BM25 ranks by **exact token overlap** weighted by term frequency and document rarity. The token `X200` matches `X200` — not `X100`. This is exactly right for model numbers, legal article citations, drug names, and any other high-precision identifier.

**But it fails on paraphrases:** the query `"can I get a refund?"` shares no tokens with the document `"return policy allows returns within 30 days"` — BM25 returns nothing. Vector search handles this trivially.

### The Hybrid Insight

Run both retrievers, then **fuse** the ranked lists. Documents relevant under either signal surface to the top. This is why every production-grade search system (Elasticsearch, Pinecone, Weaviate, OpenSearch) implements hybrid retrieval.

---

### Retrieval Strategy Comparison

| Property | BM25 (Sparse) | Vector (Dense) | Hybrid |
|----------|---------------|----------------|--------|
| Representation | Bag of words (TF-IDF weights) | Continuous vector (1536 dims) | Both |
| Exact token match | Excellent | Poor | Excellent |
| Semantic paraphrase | Poor | Excellent | Excellent |
| Out-of-vocabulary terms | Good (treats as new token) | Struggles | Good |
| Compute cost | Very low (inverted index lookup) | Medium (ANN search) | Medium |
| Best for | Model codes, names, legal refs | Conceptual queries, Q&A | Production RAG (default choice) |

### Hybrid Retrieval Architecture

```
INDEXING (run once when documents change)
──────────────────────────────────────────────────────────────────────

  Your documents
       │
       ├─────────────────────┬──────────────────────────────
       ▼                     ▼
  ┌──────────────┐     ┌──────────────┐
  │ BM25 Indexer │     │ Embed Model  │
  │ (rank_bm25)  │     │ text-emb-3   │
  └──────┬───────┘     └──────┬───────┘
         │                    │
         ▼                    ▼
  ┌──────────────┐     ┌──────────────┐
  │ Inverted     │     │ Vector Store │
  │ Index        │     │ (ChromaDB)   │
  │ (TF-IDF wts) │     │              │
  └──────────────┘     └──────────────┘


RETRIEVAL (every user query)
──────────────────────────────────────────────────────────────────────

  User query: "What is the read speed of the Apex-X200?"
       │
       ├─────────────────────────┬─────────────────────────
       ▼                         ▼
  ┌──────────────┐         ┌──────────────┐
  │ BM25         │         │ Vector       │
  │ Retriever    │         │ Retriever    │
  │ keyword rank │         │ cosine sim   │
  └──────┬───────┘         └──────┬───────┘
         │ ranked list 1           │ ranked list 2
         └──────────┬──────────────┘
                    ▼
         ┌─────────────────────┐
         │  Reciprocal Rank    │   score(d) = Σ 1/(rank_i(d) + 60)
         │  Fusion (RRF)       │   (Cormack et al., 2009)
         └──────────┬──────────┘
                    │ single fused ranked list
                    ▼
         ┌─────────────────────┐
         │  Top-k Documents    │   (optional: cross-encoder reranker)
         └──────────┬──────────┘
                    │
                    ▼
         ┌─────────────────────┐
         │  Prompt + LLM       │   generates grounded answer
         └──────────┬──────────┘
                    │
                    ▼
  "The Apex-X200 has a 7,000 MB/s sequential read speed..."
```

> **Source:** Architecture inspired by Cormack et al. (2009) RRF and the LangChain `EnsembleRetriever` implementation.

---

## Part 2 — BM25 Deep Dive: How Keyword Ranking Works

---

BM25 (Best Match 25) is a bag-of-words ranking function that scores documents by how well they match a query, considering:

1. **Term Frequency (TF)** — how many times the query token appears in the document (with diminishing returns)
2. **Inverse Document Frequency (IDF)** — how rare the token is across the corpus (rarer = more informative)
3. **Document length normalization** — penalises very long documents that match by chance

### The BM25 Formula

For query `Q` with terms `q_1 ... q_n` and document `D`:

```
BM25(D, Q) = Σ IDF(q_i) ×  TF(q_i, D) × (k1 + 1)
                            ────────────────────────────────────────────
                            TF(q_i, D) + k1 × (1 - b + b × |D| / avgdl)

Where:
  TF(q_i, D)  = count of term q_i in document D
  IDF(q_i)    = log( (N - n(q_i) + 0.5) / (n(q_i) + 0.5) + 1 )
  N           = total number of documents in corpus
  n(q_i)      = number of documents containing q_i
  |D|         = length of document D (in tokens)
  avgdl       = average document length across corpus
  k1          = term frequency saturation parameter (default 1.5)
  b           = length normalization parameter (default 0.75)
```

### BM25 Parameter Guide

| Parameter | Default | Effect of increasing | Effect of decreasing |
|-----------|---------|---------------------|---------------------|
| `k1` | 1.5 | TF has more influence (no saturation cap) | TF saturates faster (binary-like) |
| `b` | 0.75 | Longer docs penalised more | Length normalization weakened |
| `k` (top-k) | 3–10 | More candidates (more recall, more noise) | Fewer but more precise candidates |

In [ ]:
# ─── 2-A: BM25 from first principles ─────────────────────────────────────────
# Manual implementation to show the math before using the library.
# rank_bm25 does the same thing under the hood.


def bm25_score(
    query_tokens: List[str],
    doc_tokens: List[str],
    corpus_token_lists: List[List[str]],
    k1: float = 1.5,
    b: float = 0.75,
) -> float:
    """Compute BM25 score for one (query, document) pair."""
    N = len(corpus_token_lists)
    avgdl = sum(len(d) for d in corpus_token_lists) / N
    doc_len = len(doc_tokens)
    doc_tf = Counter(doc_tokens)

    score = 0.0
    for term in set(query_tokens):
        n_docs_with_term = sum(1 for d in corpus_token_lists if term in d)
        if n_docs_with_term == 0:
            continue
        idf = math.log((N - n_docs_with_term + 0.5) / (n_docs_with_term + 0.5) + 1)
        tf = doc_tf.get(term, 0)
        tf_norm = tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len / avgdl))
        score += idf * tf_norm
    return score


# Mini corpus to illustrate the formula
mini_corpus = [
    "the apex x200 has fast read speed and nvme interface",
    "the apex x100 is entry level with slower speed",
    "the vertex pro gpu has ray tracing support",
    "return policy allows returns within thirty days",
]
corpus_tokens = [doc.lower().split() for doc in mini_corpus]

query = "apex x200 speed"
query_tokens = query.lower().split()

print(f"Query: '{query}'\n")
scores = []
for i, (doc, doc_toks) in enumerate(zip(mini_corpus, corpus_tokens)):
    s = bm25_score(query_tokens, doc_toks, corpus_tokens)
    scores.append((s, doc))
    print(f"  Doc {i}: score={s:.3f}  '{doc[:60]}'")

scores.sort(reverse=True)
print(f"\n  Highest ranked: '{scores[0][1]}' (score={scores[0][0]:.3f})")

In [ ]:
# ─── 2-B: BM25Retriever from langchain_community ─────────────────────────────
# In practice we use rank_bm25 via LangChain's BM25Retriever wrapper.
# This is what EnsembleRetriever uses internally.

K = 3  # top-k documents to retrieve per retriever

# Product catalog — model numbers make this ideal for BM25 demonstration.
# Embedding models treat Apex-X200 and Apex-X100 as very similar (same sentence structure).
# BM25 treats them as completely different tokens.
DOCS = [
    "The Apex-X200 is a high-performance SSD with 7,000 MB/s sequential read speed and NVMe Gen4 interface.",
    "The Apex-X100 is an entry-level SSD with 3,500 MB/s sequential read speed and NVMe Gen3 interface.",
    "The Apex-M50 is a mechanical hard drive with 7,200 RPM and 256 MB cache, designed for archival storage.",
    "The Vertex-Pro GPU supports ray tracing, DLSS 3.0, and has 16 GB of GDDR6X memory.",
    "The Vertex-Core GPU is an entry model with 8 GB GDDR6 memory, targeting 1080p gaming.",
    "Our return policy allows returns within 30 days of purchase for all products in original packaging.",
    "Warranty coverage: Apex-X200 and Apex-X100 carry a 5-year limited warranty. Apex-M50 carries 3 years.",
    "The Apex-X200 is compatible with PCIe 4.0 and PCIe 5.0 motherboards via backward compatibility.",
    "Vertex-Pro requires a 750W PSU minimum. Vertex-Core requires 550W minimum.",
    "Technical support for Apex series products is available 24/7 via support.apextech.com.",
]

bm25_retriever = BM25Retriever.from_texts(DOCS)
bm25_retriever.k = K

exact_query = "What is the read speed of the Apex-X200?"
bm25_results = bm25_retriever.invoke(exact_query)

print(f"BM25 results for: '{exact_query}'\n")
for i, doc in enumerate(bm25_results, 1):
    print(f"  [{i}] {doc.page_content}")

In [ ]:
# ─── 2-C: BM25 blind spot — paraphrase query ──────────────────────────────────
# Demonstrate where BM25 fails: queries that use different words than the documents.

paraphrase_query = "Which product is best for budget gaming?"
bm25_paraphrase_results = bm25_retriever.invoke(paraphrase_query)

print(f"BM25 results for: '{paraphrase_query}'\n")
if bm25_paraphrase_results:
    for i, doc in enumerate(bm25_paraphrase_results, 1):
        print(f"  [{i}] {doc.page_content}")
else:
    print("  (no results — BM25 found no token overlap with this paraphrase)")

print()
print("Observation:")
print("  'budget gaming' shares no tokens with 'entry model' or '1080p gaming'")
print("  BM25 either returns irrelevant results or nothing.")
print("  Vector search handles this — we compare side by side in Part 4.")

---

## Part 3 — Vector Retrieval Recap

---

If you completed [12-basic-rag-notebook](../12-basic-rag-notebook/rag_workbook.ipynb), this is a quick recap. We build a ChromaDB vector store on the same 10 documents and test the same queries to set up the side-by-side comparison in Part 4.

### Why Embedding Models Confuse Model Numbers

```
"Apex-X200 is a high-performance SSD"  →  [0.21, -0.44, 0.78, ...]  (1536 floats)
"Apex-X100 is an entry-level SSD"      →  [0.20, -0.43, 0.77, ...]  ← almost identical!
"budget gaming GPU"                     →  [-0.12, 0.31, -0.55, ...] ← far away
```

Embedding models learn that `X200` and `X100` are product model codes for similar device types — they appear in near-identical sentence structures. In vector space they cluster together. BM25 treats them as completely different tokens.

This is not a bug in the embedding model — it is a fundamental property of learned semantic representations. Hybrid search is the correct architectural response.

In [ ]:
# ─── 3-A: Build the vector store ──────────────────────────────────────────────

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_texts(
    DOCS,
    embeddings_model,
    collection_name="hybrid_workshop",
)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": K})

print(f"Vector store loaded: {vectorstore._collection.count()} documents")

In [ ]:
# ─── 3-B: Vector retrieval on exact vs semantic queries ───────────────────────

exact_query = "What is the read speed of the Apex-X200?"
paraphrase_query = "Which product is best for budget gaming?"

print("=== Vector results — EXACT identifier query ===")
print(f"Query: '{exact_query}'\n")
vec_exact = vector_retriever.invoke(exact_query)
for i, doc in enumerate(vec_exact, 1):
    print(f"  [{i}] {doc.page_content}")

print()
print("=== Vector results — SEMANTIC/PARAPHRASE query ===")
print(f"Query: '{paraphrase_query}'\n")
vec_semantic = vector_retriever.invoke(paraphrase_query)
for i, doc in enumerate(vec_semantic, 1):
    print(f"  [{i}] {doc.page_content}")

print()
print("Observation:")
print("  Exact query: vector may surface X200 and X100 together (similar embeddings)")
print("  Semantic query: vector handles 'budget gaming' → 'entry model 1080p' well")

---

## Part 4 — Head-to-Head: BM25 vs Vector on the Same Queries

---

This is the core diagnostic. We run five annotated queries through both retrievers and compare which documents each surfaces. After this section you'll have concrete intuition for when to trust each strategy.

In [ ]:
# ─── 4-A: Side-by-side comparison on all sample queries ───────────────────────
# Each query is annotated with the expected winning strategy.

SAMPLE_QUESTIONS = [
    ("What is the read speed of the Apex-X200?",   "BM25 wins: exact model code"),
    ("Which product is best for budget gaming?",    "Vector wins: semantic intent"),
    ("What warranty does the Apex-M50 come with?",  "BM25 wins: exact model code"),
    ("How much power does the Vertex-Pro need?",    "Hybrid: model name + power concept"),
    ("Can I return a product if I change my mind?", "Vector wins: paraphrase of return policy"),
]

for question, expected_winner in SAMPLE_QUESTIONS:
    bm25_docs = bm25_retriever.invoke(question)
    vec_docs  = vector_retriever.invoke(question)

    bm25_top = bm25_docs[0].page_content[:70] if bm25_docs else "(no result)"
    vec_top  = vec_docs[0].page_content[:70]  if vec_docs  else "(no result)"

    print(f"Q: {question}")
    print(f"   Expected: [{expected_winner}]")
    print(f"   BM25 #1:  {bm25_top}...")
    print(f"   VEC  #1:  {vec_top}...")
    print()

---

## Part 5 — Reciprocal Rank Fusion: The Math

---

**Reciprocal Rank Fusion (RRF)** is a rank aggregation method proposed by Cormack et al. (2009). It is simple, robust, and remarkably parameter-insensitive — the same formula works well across diverse retrieval scenarios without tuning.

### The Formula

Given ranked lists `R_1, R_2, ..., R_m`, the RRF score for document `d` is:

```
RRF(d) =  Σ       1 / (k + rank_i(d))
         i=1..m

Where:
  rank_i(d)  = rank of document d in list i  (1-indexed; omit if doc not in list)
  k          = smoothing constant = 60  (Cormack et al.'s recommended value)
  m          = number of ranked lists
```

### Why k=60?

The constant `k=60` smooths the score curve. Without it, rank 1 would dominate catastrophically. With `k=60`, the difference between rank 1 and rank 2 is modest, and documents at rank 60 still contribute meaningfully.

### Score Intuition

```
Rank 1  in BOTH lists:    1/61 + 1/61   = 0.0328
Rank 1  in ONE list only: 1/61          = 0.0164
Rank 10 in BOTH lists:    1/70 + 1/70   = 0.0286  ← beats rank-1-only!
Rank 100 in ONE list:     1/160         = 0.0063
```

**Key insight:** Consensus across retrievers matters more than dominance in one. A document that is relevant to both BM25 and vector signals will surface even if neither ranked it first.

In [ ]:
# ─── 5-A: Manual RRF implementation ───────────────────────────────────────────
# Implementing RRF from scratch so the math is concrete before using EnsembleRetriever.


def reciprocal_rank_fusion(
    ranked_lists: List[List[str]],
    k: int = 60,
) -> List[tuple]:
    """Fuse multiple ranked document lists using RRF. Returns (doc, score) sorted descending."""
    scores: dict = {}
    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):  # 1-indexed
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# Simulate two ranked lists for the Apex-X200 query
bm25_ranked   = [
    "Apex-X200 (7000 MB/s read)",
    "Apex-X200 PCIe compat",
    "Apex-X100 (3500 MB/s)",
    "Vertex-Core 1080p gaming",
]
vector_ranked = [
    "Apex-X100 (3500 MB/s)",
    "Apex-X200 (7000 MB/s read)",
    "Apex-X200 PCIe compat",
    "Warranty Apex-X200 X100",
]

fused = reciprocal_rank_fusion([bm25_ranked, vector_ranked])

print("RRF fusion of BM25 + Vector ranked lists\n")
print(f"{'Rank':<6} {'Score':<8} {'Document':<34} {'BM25 pos':>9} {'Vec pos':>8}")
print("-" * 70)
for rank, (doc, score) in enumerate(fused, 1):
    bm25_pos = bm25_ranked.index(doc) + 1 if doc in bm25_ranked else "-"
    vec_pos  = vector_ranked.index(doc) + 1 if doc in vector_ranked else "-"
    print(f"{rank:<6} {score:.4f}   {doc:<34} {str(bm25_pos):>9} {str(vec_pos):>8}")

print()
print("Key insight: 'Apex-X200' appears in BOTH lists → highest combined RRF score")

In [ ]:
# ─── 5-B: RRF score curve — visualise rank vs score contribution ───────────────
# Shows why k=60 prevents rank 1 from catastrophically dominating.

print("RRF score contribution by rank (k=60)\n")
print(f"{'Rank':<8} {'Score (1 list)':<18} {'Score (2 lists)':<18} {'Bar'}")
print("-" * 65)

for rank in [1, 2, 3, 5, 10, 20, 50, 100]:
    single = 1 / (60 + rank)
    both   = 2 / (60 + rank)
    bar    = "█" * int(single * 1000)
    print(f"{rank:<8} {single:.5f}            {both:.5f}            {bar}")

print()
print("rank 1 in ONE list:    0.01639")
print("rank 10 in BOTH lists: 0.02857  ← consensus at rank 10 beats lone rank 1")

---

## Part 6 — EnsembleRetriever: LangChain's Hybrid Wrapper

---

`EnsembleRetriever` from `langchain.retrievers` implements RRF automatically. You pass a list of retrievers and weights — the weights scale RRF scores before merging (not the raw document scores directly).

```python
ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],  # equal weight — tune per domain
)
```

### Weight Tuning Guide

| Domain | Recommended weights | Rationale |
|--------|--------------------|-----------|
| Product catalog / e-commerce | `[0.7, 0.3]` BM25 higher | Model numbers, SKUs need exact match |
| Legal / medical documents | `[0.6, 0.4]` BM25 higher | Precise terminology, article references |
| Customer support / FAQ | `[0.4, 0.6]` Vector higher | Users paraphrase; intent matters |
| General Q&A / knowledge base | `[0.5, 0.5]` Equal | Balanced starting point |
| Conversational / conceptual | `[0.3, 0.7]` Vector higher | Semantic intent dominates |

In [ ]:
# ─── 6-A: Build the EnsembleRetriever ────────────────────────────────────────

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

print("EnsembleRetriever ready (BM25 + Vector, weights [0.5, 0.5])\n")

for question, expected_winner in SAMPLE_QUESTIONS:
    docs = ensemble_retriever.invoke(question)
    top  = docs[0].page_content[:80] if docs else "(no result)"
    print(f"Q: {question}")
    print(f"   [{expected_winner}]")
    print(f"   Hybrid #1: {top}...")
    print()

In [ ]:
# ─── 6-B: Effect of weight tuning ─────────────────────────────────────────────
# Show how changing weights affects retrieval on both query types.

semantic_q = "Which product is best for budget gaming?"
exact_q    = "What is the read speed of the Apex-X200?"

print("Weight sensitivity experiment\n")
print(f"Semantic query: '{semantic_q}'")
print(f"Exact query:    '{exact_q}'")
print()

weight_configs = [
    ([0.8, 0.2], "BM25-heavy"),
    ([0.5, 0.5], "Balanced  "),
    ([0.2, 0.8], "Vec-heavy "),
]

for weights, label in weight_configs:
    ens = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=weights)
    sem_top = ens.invoke(semantic_q)[0].page_content[:65]
    ex_top  = ens.invoke(exact_q)[0].page_content[:65]
    print(f"  {label} {weights}:")
    print(f"    Semantic top: {sem_top}...")
    print(f"    Exact    top: {ex_top}...")
    print()

---

## Part 7 — Full RAG Pipeline: LangGraph Retrieve → Generate

---

We wire the `EnsembleRetriever` into a two-node LangGraph pipeline:

```
START → [retrieve] → [generate] → END

  retrieve:  EnsembleRetriever.invoke(question) → top-k fused documents
  generate:  LLM answers from context only — refuses to hallucinate
```

The `HybridRAGState` TypedDict flows through the graph, accumulating fields at each node. This matches the pattern in `src/workflow.py` — the workbook unpacks every step that the production module encapsulates.

In [ ]:
# ─── 7-A: Define state and build the graph ────────────────────────────────────

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class HybridRAGState(TypedDict):
    question: str
    documents: list   # list[str] — retrieved doc contents
    answer: str


def retrieve(state: HybridRAGState) -> dict:
    """Fetch top-k documents via EnsembleRetriever (BM25 + vector, RRF fusion)."""
    docs = ensemble_retriever.invoke(state["question"])
    return {"documents": [d.page_content for d in docs]}


def generate(state: HybridRAGState) -> dict:
    """Generate an answer from retrieved context only — no hallucination."""
    context = "\n\n".join(state["documents"])
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a helpful product support assistant. "
                    "Answer using ONLY the context below. "
                    "If the answer is not in the context, say 'I don't have that information.'\n\n"
                    f"Context:\n{context}"
                )
            ),
            HumanMessage(content=state["question"]),
        ]
    )
    return {"answer": response.content}


graph = StateGraph(HybridRAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
app = graph.compile()

print("LangGraph compiled: START → retrieve → generate → END")

In [ ]:
# ─── 7-B: Run the full pipeline on all sample questions ───────────────────────

for question, expected_winner in SAMPLE_QUESTIONS:
    result = app.invoke({"question": question, "documents": [], "answer": ""})
    print(f"Q: {question}")
    print(f"   [{expected_winner}]")
    print(f"   A: {result['answer']}")
    print(f"   Retrieved {len(result['documents'])} docs")
    print()

In [ ]:
# ─── 7-C: Inspect retrieved context — the #1 RAG debugging technique ──────────
# ALWAYS check what was retrieved before trusting an LLM answer.

debug_question = "What is the read speed of the Apex-X200?"
result = app.invoke({"question": debug_question, "documents": [], "answer": ""})

print(f"Q: {debug_question}")
print(f"A: {result['answer']}")
print()
print("Retrieved context (what the LLM actually saw):")
for i, doc in enumerate(result["documents"], 1):
    print(f"  [{i}] {doc}")

---

## Part 8 ★ — Advanced: Reranking on Top of Hybrid Retrieval

---

Hybrid retrieval with RRF is the standard **first-stage retriever** (high recall, moderate precision). For higher-precision use cases, add a **cross-encoder reranker** as a second stage.

### Two-Stage Retrieval

```
Stage 1 — Recall (fast, approximate)
──────────────────────────────────────
  BM25 + Vector → RRF → top-50 candidates
  (bi-encoders: fast pre-computed vectors, cosine similarity)

Stage 2 — Precision (slower, exact)
──────────────────────────────────────
  Cross-encoder scores every (query, candidate) pair jointly
  Returns re-ranked top-5
  (query + doc encoded together → direct relevance score)
```

### Bi-encoder vs Cross-encoder

| | Bi-encoder (embedding model) | Cross-encoder |
|--|------------------------------|---------------|
| Encodes | Query and doc separately | Query + doc together |
| Similarity | Cosine similarity of separate vectors | Direct relevance score |
| Speed | Fast (pre-compute doc embeddings) | Slow (pair computed at runtime) |
| Accuracy | Good | Better — full cross-attention context |
| Pipeline role | First-stage retrieval (candidate generation) | Second-stage reranking |

### Reranking Strategies

| Strategy | Library | Notes |
|----------|---------|-------|
| Cross-encoder | `sentence-transformers` + `CrossEncoderReranker` | Best quality, local |
| Cohere Rerank | `langchain_cohere` | API-based, no local model needed |
| FlashRank | `flashrank` + `FlashrankRerank` | Fast local reranker |
| LLM-as-judge | `ContextualCompressionRetriever` + custom LLM | Most flexible, most expensive |

```python
# Cross-encoder reranking with LangChain
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import CrossEncoderReranker
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

# Wrap the ensemble retriever — fetch 20, rerank to 3
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=ensemble_retriever,
)
```

> **Note:** Cross-encoders require `sentence-transformers` (`pip install sentence-transformers`). Add it to requirements.txt when reranking is needed in production.

---

## Exercises

---

### Exercise 1 — Tune the BM25/Vector Balance

Change `weights` in the cell below and re-run all five `SAMPLE_QUESTIONS`. Try `[0.7, 0.3]` (BM25-heavy), `[0.5, 0.5]` (balanced), and `[0.3, 0.7]` (vector-heavy).

**Questions to answer:**
- Which weight config best handles exact model codes like `"Apex-X200"`?
- Which best handles the `"budget gaming"` paraphrase?
- Where does balanced `[0.5, 0.5]` fall short compared to a tuned split?

### Exercise 2 — BM25 Blind Spot Recovery

Remove `"Vertex-Core"` from DOCS and rebuild both retrievers. Ask `"What is the entry-level GPU?"` through BM25 alone, vector alone, and hybrid. Which still finds the right result when the exact name is gone?

### Exercise 3 — Your Own Domain

Replace DOCS with 10+ sentences from a domain you know (medical terminology, legal references, internal engineering docs, product catalog). Write:
- 2 queries where BM25 should win (precise identifiers)
- 2 queries where vector should win (conceptual paraphrases)
- 1 query that needs both signals

Verify hybrid covers all five.

In [ ]:
# ── Exercise 1 Starter: Tune weights ─────────────────────────────────────────

# TODO: change these weights to [0.7, 0.3] and [0.3, 0.7] and compare
MY_WEIGHTS = [0.5, 0.5]

my_ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=MY_WEIGHTS,
)

test_queries = [
    "What is the read speed of the Apex-X200?",   # BM25 advantage
    "Which product is best for budget gaming?",    # Vector advantage
]

print(f"Testing weights: {MY_WEIGHTS}\n")
for q in test_queries:
    docs = my_ensemble.invoke(q)
    print(f"Q: {q}")
    print(f"   Top result: {docs[0].page_content[:80]}..." if docs else "   (no result)")
    print()

In [ ]:
# ── Exercise 2 Starter: Remove a product name, test recovery ──────────────────

# TODO: Fill in DOCS_MODIFIED — copy DOCS but remove the Vertex-Core line
DOCS_MODIFIED = [
    # paste all DOCS entries here except the Vertex-Core line
]

if DOCS_MODIFIED:
    bm25_mod = BM25Retriever.from_texts(DOCS_MODIFIED)
    bm25_mod.k = K
    vec_mod = Chroma.from_texts(
        DOCS_MODIFIED, embeddings_model, collection_name="ex2"
    ).as_retriever(search_kwargs={"k": K})
    ens_mod = EnsembleRetriever(retrievers=[bm25_mod, vec_mod], weights=[0.5, 0.5])

    q = "What is the entry-level GPU?"
    print(f"Q: {q}  (Vertex-Core removed from corpus)\n")
    bm25_r = bm25_mod.invoke(q)
    vec_r  = vec_mod.invoke(q)
    ens_r  = ens_mod.invoke(q)
    print("  BM25:  ", bm25_r[0].page_content[:80] if bm25_r else "(no result)")
    print("  VEC:   ", vec_r[0].page_content[:80]  if vec_r  else "(no result)")
    print("  Hybrid:", ens_r[0].page_content[:80]  if ens_r  else "(no result)")
else:
    print("TODO: Fill in DOCS_MODIFIED above")

In [ ]:
# ── Exercise 3 Starter: Your own domain ───────────────────────────────────────

MY_DOCS = [
    # TODO: add 10+ sentences from your domain
    # Example:
    # "Product ABC-1000 has a 2-year warranty and 50 GB storage.",
    # "Our refund window is 14 calendar days from the delivery date.",
]

MY_QUERIES = [
    # TODO: 2 BM25-wins (precise identifiers), 2 vector-wins (paraphrases), 1 hybrid
    # "Query using exact identifier ABC-1000",
    # "Conceptual query without exact keywords",
]

if MY_DOCS and MY_QUERIES:
    my_bm25 = BM25Retriever.from_texts(MY_DOCS)
    my_bm25.k = K
    my_vec = Chroma.from_texts(
        MY_DOCS, embeddings_model, collection_name="exercise3"
    ).as_retriever(search_kwargs={"k": K})
    my_ens = EnsembleRetriever(retrievers=[my_bm25, my_vec], weights=[0.5, 0.5])

    for q in MY_QUERIES:
        docs = my_ens.invoke(q)
        print(f"Q: {q}")
        print(f"   Top: {docs[0].page_content[:80]}..." if docs else "   (no result)")
        print()
else:
    print("TODO: Fill in MY_DOCS and MY_QUERIES above")

---

## What's Next?

You now understand the full hybrid retrieval stack — BM25, dense vector search, RRF fusion, and the LangGraph RAG pipeline. Here's where to go deeper:

### Evaluate your retrieval quality
- **Example 16 — RAG Evaluation with RAGAS** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): score Faithfulness, Answer Relevance, and Context Recall — measure whether hybrid actually outperforms vector-only on your dataset. Run a weight sweep `[0.3, 0.7]` to `[0.7, 0.3]` and let RAGAS pick the best split.

### Different angles on multi-source retrieval
- **Example 26 — RAG Fusion** ([`../26-rag-fusion/`](../26-rag-fusion/)): RRF applied to *query variants* instead of retriever variants — generate 5 paraphrases of the query, retrieve for each, fuse with RRF
- **Example 25 — Adaptive RAG** ([`../25-adaptive-rag/`](../25-adaptive-rag/)): route queries to the right strategy (hybrid, vector-only, keyword-only) based on query classification — avoid hybrid overhead when a simpler strategy suffices

### Multi-hop and graph-based retrieval
- **Example 24 — Graph RAG** ([`../24-graph-rag/`](../24-graph-rag/)): when even hybrid search misses multi-hop relational queries — entities connected across documents

### Production considerations
- **BM25 at scale:** `BM25Retriever` is in-memory and rebuilt each run. For persistent BM25 at production scale, use Elasticsearch or OpenSearch with BM25+kNN hybrid mode
- **Sparse neural vectors:** SPLADE and ColBERT learn sparse neural representations — the next frontier beyond lexical BM25, with learned term expansion
- **Latency:** BM25 lookup is microseconds; vector ANN search is milliseconds; both run in parallel in `EnsembleRetriever`

---

*Workshop complete. Next step: open example 16 and put a RAGAS score on the retrieval quality you just built.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are sample solutions — not the only correct answers.

---

### Exercise 1 — Tune the BM25/Vector Balance

**Expected findings:**

- `[0.7, 0.3]` BM25-heavy: `"Apex-X200 read speed"` surfaces the exact doc at rank 1 reliably; `"budget gaming"` may not find the Vertex-Core because BM25 finds no token overlap with that paraphrase
- `[0.3, 0.7]` vector-heavy: `"budget gaming"` correctly surfaces the entry-level GPU; `"Apex-X200 read speed"` may confuse X200 with X100 since their embeddings are very close
- `[0.5, 0.5]` balanced: best all-around performance on this mixed corpus — neither retriever dominates inappropriately

**Rule of thumb:** Run RAGAS on a labeled eval set with each weight config rather than guessing. The optimal split is domain-specific.

In [ ]:
# Answer key — Exercise 1: programmatic weight comparison

eval_queries = [
    ("What is the read speed of the Apex-X200?",   "Apex-X200",   "BM25-type"),
    ("Which product is best for budget gaming?",    "Vertex-Core", "Vec-type"),
    ("Can I return a product if I change my mind?", "return",      "Vec-type"),
    ("What warranty does the Apex-M50 come with?",  "Apex-M50",    "BM25-type"),
]

weight_configs = [
    ([0.7, 0.3], "BM25-heavy"),
    ([0.5, 0.5], "Balanced  "),
    ([0.3, 0.7], "Vec-heavy "),
]

print(f"{'Config':<22} {'BM25-type correct':>18} {'Vec-type correct':>17}")
print("-" * 60)

for weights, label in weight_configs:
    ens = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=weights)
    bm25_hits, vec_hits = 0, 0
    bm25_total, vec_total = 0, 0

    for q, expected_token, query_type in eval_queries:
        docs = ens.invoke(q)
        top_content = docs[0].page_content if docs else ""
        hit = expected_token.lower() in top_content.lower()
        if query_type == "BM25-type":
            bm25_total += 1
            if hit:
                bm25_hits += 1
        else:
            vec_total += 1
            if hit:
                vec_hits += 1

    print(f"  {label} {weights}   {bm25_hits}/{bm25_total} BM25-type   {vec_hits}/{vec_total} Vec-type")

### Exercise 2 — BM25 Blind Spot Recovery

**Expected finding:** When `Vertex-Core` is removed from DOCS:
- BM25 returns zero results or unrelated docs for `"entry-level GPU"` — no token overlap with remaining documents
- Vector search still surfaces `Vertex-Pro` because the embedding for `"entry-level GPU"` is semantically close to GPU-related documents even without the exact phrase
- Hybrid returns a useful result because the vector signal rescues what BM25 misses

**Key lesson:** Hybrid is more robust to corpus gaps than either retriever alone. This matters in production where your corpus is never perfectly complete.

In [ ]:
# Answer key — Exercise 2: remove Vertex-Core and compare all three retrievers

DOCS_NO_CORE = [d for d in DOCS if "Vertex-Core" not in d]

bm25_nc = BM25Retriever.from_texts(DOCS_NO_CORE)
bm25_nc.k = K
vec_nc  = Chroma.from_texts(
    DOCS_NO_CORE, embeddings_model, collection_name="answer_key_ex2"
).as_retriever(search_kwargs={"k": K})
ens_nc  = EnsembleRetriever(retrievers=[bm25_nc, vec_nc], weights=[0.5, 0.5])

q = "What is the entry-level GPU?"
print(f"Q: '{q}'  (Vertex-Core removed from corpus)\n")

bm25_r = bm25_nc.invoke(q)
vec_r  = vec_nc.invoke(q)
ens_r  = ens_nc.invoke(q)

print(f"  BM25:   {bm25_r[0].page_content[:80] if bm25_r else '(no result)'}")
print(f"  Vector: {vec_r[0].page_content[:80]  if vec_r  else '(no result)'}")
print(f"  Hybrid: {ens_r[0].page_content[:80]  if ens_r  else '(no result)'}")
print()
print("Observation: Hybrid (via vector signal) still finds GPU-related content even without exact token match.")

---

## Academic References

1. **Robertson, S. & Walker, S. (1994).** *Some Simple Effective Approximations to the 2-Poisson Model for Probabilistic Weighted Retrieval.* Proceedings of SIGIR 1994. *(Original BM25 formulation)*

2. **Robertson, S. & Zaragoza, H. (2009).** *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in Information Retrieval, 3(4), 333–389. *(Comprehensive BM25 reference including the k1/b parameter analysis)*

3. **Cormack, G.V., Clarke, C.L.A., & Buettcher, S. (2009).** *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods.* Proceedings of SIGIR 2009. https://dl.acm.org/doi/10.1145/1571941.1572114 *(The RRF paper; establishes k=60 as the recommended constant)*

4. **Lewis, P., Perez, E., et al. (2020).** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* Advances in Neural Information Processing Systems (NeurIPS) 2020. https://arxiv.org/abs/2005.11401 *(Foundational RAG paper from Meta AI)*

5. **Gao, Y., Xiong, Y., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey.* https://arxiv.org/abs/2312.10997 *(Survey covering hybrid retrieval, reranking, and production patterns)*

6. **Formal, T., Piwowarski, B., & Clinchant, S. (2021).** *SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking.* SIGIR 2021. *(Neural sparse retrieval — the next evolution beyond lexical BM25)*